# MAC vs DIN Dataset Run Figures

Notebook này gom thống kê các run đã có cho MAC-SQL và DIN-SQL trên các dataset lớn đã chạy: Spider dev, Spider train_spider, Spider train_others, và BIRD dev.

Các figure được tạo trực tiếp từ file result hiện có trong repo, không hard-code accuracy. Output PNG sẽ được lưu trong `baselines/text2sql/analysis/figures_mac_din/`.


In [ ]:
from __future__ import annotations

import json
import math
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    """Find repository root even when Jupyter starts in the notebook folder."""
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'baselines/text2sql/04_din_sql').exists() and (candidate / 'baselines/text2sql/06_mac_sql').exists():
            return candidate
    raise RuntimeError(f'Cannot find repo root from {start}. Start Jupyter inside /mnt/c/research/mas/paper/scm or set ROOT manually.')

ROOT = find_repo_root()
OUT_DIR = ROOT / 'baselines/text2sql/analysis/figures_mac_din'
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (10, 5.5),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

DATASET_TOTALS = {
    'spider_dev': 1034,
    'spider_train_spider': 7000,
    'spider_train_others': 1659,
    'bird_dev': 1534,
}

print('root =', ROOT)
print('figures =', OUT_DIR)


In [ ]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def load_json(path: Path) -> Any:
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def is_framework_ex(row: dict[str, Any]) -> bool:
    pred = row.get('pred_exec') or {}
    gold = row.get('gold_exec') or {}
    return pred.get('status') == 'ok' and gold.get('status') == 'ok' and pred.get('result_hash') == gold.get('result_hash')


def row_idx(row: dict[str, Any]) -> Any:
    case = row.get('case') or {}
    return case.get('idx', row.get('idx', row.get('question_id')))


def dedupe_by_idx(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_idx = {}
    no_idx = []
    for row in rows:
        idx = row_idx(row)
        if idx is None:
            no_idx.append(row)
        else:
            by_idx[idx] = row
    return list(by_idx.values()) + no_idx


def pick_prediction_file(run_dir: Path) -> Path | None:
    final = run_dir / 'predictions.jsonl'
    partial = run_dir / 'predictions.partial.jsonl'
    if final.exists():
        return final
    if partial.exists():
        return partial
    return None


def summarize_framework_jsonl(
    baseline: str,
    dataset: str,
    paths: list[Path],
    total_expected: int,
    note: str = '',
) -> tuple[dict[str, Any], pd.DataFrame]:
    rows = []
    for path in paths:
        rows.extend(load_jsonl(path))
    rows = dedupe_by_idx(rows)
    correct = sum(is_framework_ex(row) for row in rows)
    completed = len(rows)
    missing = max(0, total_expected - completed)
    pred_ok = sum((row.get('pred_exec') or {}).get('status') == 'ok' for row in rows)
    call_errors = sum(bool(row.get('call_error')) for row in rows)
    empty_sql = sum(not (row.get('pred_sql') or '').strip() for row in rows)
    pred_exec_errors = sum((row.get('pred_exec') or {}).get('status') not in ('ok', None, '') for row in rows)
    summary = {
        'baseline': baseline,
        'dataset': dataset,
        'total_expected': total_expected,
        'completed': completed,
        'missing': missing,
        'correct': correct,
        'wrong_or_error': completed - correct,
        'accuracy_completed': correct / completed if completed else None,
        'accuracy_count_missing_wrong': correct / total_expected if total_expected else None,
        'coverage': completed / total_expected if total_expected else None,
        'pred_ok': pred_ok,
        'pred_exec_errors': pred_exec_errors,
        'call_errors': call_errors,
        'empty_sql': empty_sql,
        'source': '; '.join(str(p.relative_to(ROOT)) for p in paths),
        'note': note,
    }
    detail = pd.DataFrame([
        {
            'baseline': baseline,
            'dataset': dataset,
            'idx': row_idx(row),
            'db_id': (row.get('case') or {}).get('db_id') or row.get('db_id'),
            'correct': is_framework_ex(row),
            'pred_status': (row.get('pred_exec') or {}).get('status', ''),
            'difficulty': (row.get('case') or {}).get('difficulty') or row.get('difficulty'),
            'call_error': row.get('call_error', ''),
            'pred_sql': row.get('pred_sql', ''),
            'gold_sql': row.get('gold_sql', ''),
        }
        for row in rows
    ])
    return summary, detail


In [ ]:
def summarize_mac_eval_list(
    baseline: str,
    dataset: str,
    eval_path: Path,
    total_expected: int,
    note: str = '',
) -> tuple[dict[str, Any], pd.DataFrame]:
    data = load_json(eval_path) or []
    if isinstance(data, dict):
        # BIRD conservative JSON sometimes uses a list-like file; keep this fallback explicit.
        data = data.get('results') or data.get('details') or []
    rows = data if isinstance(data, list) else []
    correct = sum(int(row.get('exec_result', row.get('res', 0)) == 1) for row in rows)
    completed = len(rows)
    missing = max(0, total_expected - completed)
    summary = {
        'baseline': baseline,
        'dataset': dataset,
        'total_expected': total_expected,
        'completed': completed,
        'missing': missing,
        'correct': correct,
        'wrong_or_error': completed - correct,
        'accuracy_completed': correct / completed if completed else None,
        'accuracy_count_missing_wrong': correct / total_expected if total_expected else None,
        'coverage': completed / total_expected if total_expected else None,
        'pred_ok': None,
        'pred_exec_errors': None,
        'call_errors': None,
        'empty_sql': None,
        'source': str(eval_path.relative_to(ROOT)),
        'note': note,
    }
    detail = pd.DataFrame([
        {
            'baseline': baseline,
            'dataset': dataset,
            'idx': row.get('idx', row.get('question_id')),
            'db_id': row.get('db_id'),
            'correct': row.get('exec_result', row.get('res', 0)) == 1,
            'pred_sql': row.get('pred', ''),
            'gold_sql': row.get('gold', ''),
            'difficulty': row.get('difficulty'),
        }
        for row in rows
    ])
    return summary, detail


def summarize_mac_robust_summary(
    baseline: str,
    dataset: str,
    summary_path: Path,
    note: str = '',
) -> tuple[dict[str, Any], pd.DataFrame]:
    data = load_json(summary_path) or {}
    total = int(data.get('total', 0))
    valid_gold = int(data.get('valid_gold', total))
    invalid_gold = int(data.get('invalid_gold', max(0, total - valid_gold)))
    correct = int(data.get('correct_on_valid_gold', 0))
    pred_error = int(data.get('pred_error_on_valid_gold', 0))
    summary = {
        'baseline': baseline,
        'dataset': dataset,
        'total_expected': total,
        'completed': total,
        'missing': 0,
        'correct': correct,
        'wrong_or_error': total - correct,
        'accuracy_completed': data.get('execution_accuracy_count_invalid_as_wrong'),
        'accuracy_count_missing_wrong': data.get('execution_accuracy_count_invalid_as_wrong'),
        'coverage': 1.0 if total else None,
        'pred_ok': valid_gold - pred_error,
        'pred_exec_errors': pred_error,
        'call_errors': None,
        'empty_sql': None,
        'source': str(summary_path.relative_to(ROOT)),
        'note': note or f'robust eval; invalid_gold={invalid_gold}',
    }
    detail = pd.DataFrame(data.get('invalid_gold_examples', []))
    return summary, detail


In [ ]:
# --- DIN result paths ---
din_spider_dev = [ROOT / 'baselines/text2sql/04_din_sql/results/smoke_spider_dev_adapted_api_n1034_cohere_rplus_20260731_full/predictions.jsonl']
din_spider_train_others = [ROOT / 'baselines/text2sql/04_din_sql/results/smoke_spider_train_others_adapted_api_n1659_cohere_rplus_20260731_full_debug/predictions.jsonl']

din_spider_train_spider_dirs = sorted((ROOT / 'baselines/text2sql/04_din_sql/results').glob('smoke_spider_train_spider_adapted_api_n875_cohere_rplus_20260802_shard_*'))
din_spider_train_spider = [p for d in din_spider_train_spider_dirs if (p := pick_prediction_file(d)) is not None]

din_bird_dirs = sorted((ROOT / 'baselines/text2sql/04_din_sql/results').glob('smoke_bird_dev_adapted_api_n*_cohere_rplus_2026080*'))
din_bird_dev = []
for d in din_bird_dirs:
    # Exclude tiny early smoke runs and prefer final file when present.
    if 'smoke2' in d.name:
        continue
    p = pick_prediction_file(d)
    if p is not None:
        din_bird_dev.append(p)

# --- MAC native result/eval paths ---
mac_base = ROOT / 'baselines/text2sql/06_mac_sql/code/MAC-SQL/outputs'
mac_spider_dev_eval = mac_base / 'spider_dev_cohere_rplus_20260727/evaluation.json'
mac_train_others_eval = mac_base / 'spider_train_others_cohere_rplus_20260727/evaluation.json'
mac_train_spider_robust = mac_base / 'spider_train_spider_cohere_rplus_20260727/eval_exec_robust_from_json_safe_gold_summary.json'
mac_bird_eval = mac_base / 'bird_dev_cohere_rplus_20260727/eval_result_dev.json'

for label, paths in {
    'din_spider_dev': din_spider_dev,
    'din_spider_train_others': din_spider_train_others,
    'din_spider_train_spider': din_spider_train_spider,
    'din_bird_dev': din_bird_dev,
}.items():
    print(label, len(paths))
    for p in paths[:5]:
        print(' ', p.relative_to(ROOT))
    if len(paths) > 5:
        print('  ...')


In [ ]:
summary_rows = []
detail_frames = []

for summary, detail in [
    summarize_framework_jsonl('DIN-SQL', 'spider_dev', din_spider_dev, DATASET_TOTALS['spider_dev']),
    summarize_framework_jsonl('DIN-SQL', 'spider_train_others', din_spider_train_others, DATASET_TOTALS['spider_train_others']),
    summarize_framework_jsonl('DIN-SQL', 'spider_train_spider', din_spider_train_spider, DATASET_TOTALS['spider_train_spider'], note='8 shard files deduplicated by idx'),
    summarize_framework_jsonl('DIN-SQL', 'bird_dev', din_bird_dev, DATASET_TOTALS['bird_dev'], note='deduplicated across BIRD full/shard/rem files'),
    summarize_mac_eval_list('MAC-SQL', 'spider_dev', mac_spider_dev_eval, DATASET_TOTALS['spider_dev']),
    summarize_mac_eval_list('MAC-SQL', 'spider_train_others', mac_train_others_eval, DATASET_TOTALS['spider_train_others']),
    summarize_mac_robust_summary('MAC-SQL', 'spider_train_spider', mac_train_spider_robust),
    summarize_mac_eval_list('MAC-SQL', 'bird_dev', mac_bird_eval, DATASET_TOTALS['bird_dev']),
]:
    summary_rows.append(summary)
    if not detail.empty:
        detail_frames.append(detail)

summary_df = pd.DataFrame(summary_rows)
expected_detail_cols = ['baseline', 'dataset', 'idx', 'db_id', 'correct', 'pred_sql', 'gold_sql', 'difficulty', 'pred_status', 'call_error']
details_df = pd.concat(detail_frames, ignore_index=True, sort=False) if detail_frames else pd.DataFrame(columns=expected_detail_cols)
for col in expected_detail_cols:
    if col not in details_df.columns:
        details_df[col] = pd.NA

summary_df['accuracy_pct'] = summary_df['accuracy_count_missing_wrong'] * 100
summary_df['coverage_pct'] = summary_df['coverage'] * 100
summary_df = summary_df.sort_values(['dataset', 'baseline']).reset_index(drop=True)

summary_df[['baseline','dataset','completed','total_expected','coverage_pct','correct','accuracy_pct','wrong_or_error','missing','note']]


In [ ]:
# Save the machine-readable summary used by the figures.
summary_csv = ROOT / 'baselines/text2sql/analysis/mac_din_dataset_run_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print('wrote', summary_csv.relative_to(ROOT))
summary_df[['baseline','dataset','completed','total_expected','correct','accuracy_pct','coverage_pct','wrong_or_error','missing']]


In [ ]:
def savefig(name: str) -> None:
    path = OUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches='tight')
    print('saved', path.relative_to(ROOT))


def annotate_bars(ax, fmt='{:.1f}'):
    for container in ax.containers:
        labels = []
        for value in container.datavalues:
            if pd.isna(value):
                labels.append('')
            else:
                labels.append(fmt.format(value))
        ax.bar_label(container, labels=labels, padding=3, fontsize=9)


In [ ]:
# Figure 1: Execution accuracy by dataset and baseline.
plot_df = summary_df.copy()
order = ['spider_dev', 'spider_train_spider', 'spider_train_others', 'bird_dev']
pivot = plot_df.pivot(index='dataset', columns='baseline', values='accuracy_pct').reindex(order)
ax = pivot.plot(kind='bar', color=['#4C78A8', '#F58518'], width=0.72)
ax.set_title('Execution Accuracy by Dataset')
ax.set_xlabel('Dataset')
ax.set_ylabel('Execution accuracy (%)')
ax.set_ylim(0, 105)
ax.legend(title='Baseline')
annotate_bars(ax)
savefig('fig1_mac_din_accuracy_by_dataset.png')


In [ ]:
# Figure 2: Completed coverage by dataset.
pivot = summary_df.pivot(index='dataset', columns='baseline', values='coverage_pct').reindex(order)
ax = pivot.plot(kind='bar', color=['#72B7B2', '#E45756'], width=0.72)
ax.set_title('Run Coverage by Dataset')
ax.set_xlabel('Dataset')
ax.set_ylabel('Completed examples (%)')
ax.set_ylim(0, 105)
ax.legend(title='Baseline')
annotate_bars(ax)
savefig('fig2_mac_din_coverage_by_dataset.png')


In [ ]:
# Figure 3: Correct / incorrect / missing composition.
comp = summary_df.copy()
comp['incorrect_or_error'] = comp['wrong_or_error']
comp['label'] = comp['baseline'] + '\n' + comp['dataset']
comp = comp.sort_values(['dataset','baseline'])
fig, ax = plt.subplots(figsize=(12, 6))
left = [0] * len(comp)
for col, color, label in [
    ('correct', '#54A24B', 'Correct'),
    ('incorrect_or_error', '#E45756', 'Wrong/error'),
    ('missing', '#BAB0AC', 'Missing'),
]:
    ax.bar(comp['label'], comp[col], bottom=left, color=color, label=label)
    left = [a + b for a, b in zip(left, comp[col])]
ax.set_title('Outcome Composition by Run')
ax.set_ylabel('Number of examples')
ax.tick_params(axis='x', rotation=45)
ax.legend()
savefig('fig3_mac_din_outcome_composition.png')


In [ ]:
# Figure 4: DIN execution-error and call-error breakdown.
din_err = summary_df[summary_df['baseline'] == 'DIN-SQL'].copy()
err_cols = ['pred_exec_errors', 'call_errors', 'empty_sql', 'missing']
for col in err_cols:
    din_err[col] = din_err[col].fillna(0)
ax = din_err.set_index('dataset')[err_cols].reindex(order).plot(kind='bar', stacked=True, color=['#E45756','#B279A2','#FF9DA6','#BAB0AC'], figsize=(10,5.5))
ax.set_title('DIN-SQL Non-success Breakdown')
ax.set_xlabel('Dataset')
ax.set_ylabel('Count')
ax.legend(title='Category')
savefig('fig4_din_error_breakdown.png')


In [ ]:
# Figure 5: BIRD difficulty breakdown when detail labels are available.
required_cols = {'dataset', 'difficulty', 'baseline', 'correct'}
if not required_cols.issubset(details_df.columns):
    print(f'Missing columns for BIRD difficulty figure: {sorted(required_cols - set(details_df.columns))}')
else:
    bird_detail = details_df[(details_df['dataset'] == 'bird_dev') & details_df['difficulty'].notna()].copy()
    if not bird_detail.empty:
        bird_summary = (
            bird_detail.groupby(['baseline','difficulty'])
            .agg(n=('correct','size'), correct=('correct','sum'))
            .reset_index()
        )
        bird_summary['accuracy_pct'] = bird_summary['correct'] / bird_summary['n'] * 100
        diff_order = ['simple', 'moderate', 'challenging']
        pivot = bird_summary.pivot(index='difficulty', columns='baseline', values='accuracy_pct').reindex(diff_order)
        ax = pivot.plot(kind='bar', color=['#4C78A8','#F58518'], width=0.72)
        ax.set_title('BIRD Dev Accuracy by Difficulty')
        ax.set_xlabel('Difficulty')
        ax.set_ylabel('Execution accuracy (%)')
        ax.set_ylim(0, 105)
        ax.legend(title='Baseline')
        annotate_bars(ax)
        savefig('fig5_bird_accuracy_by_difficulty.png')
        try:
            display(bird_summary)
        except NameError:
            print(bird_summary)
    else:
        print('No BIRD difficulty labels found in loaded details.')


In [ ]:
# Optional: inspect the exact sources used for every summary row.
pd.set_option('display.max_colwidth', 120)
summary_df[['baseline','dataset','source','note']]
